# Architecture

What is the underlying architecture of Kafi Streams?

Here is a dependency diagram:

```mermaid
---
title: Kafi Streams dependency diagram
---
classDiagram
    Streams <|-- TopologyNode
    Streams <|-- Kafi

    TopologyNode <|-- pydbsp
    TopologyNode <|-- msgpack
    TopologyNode <|-- cloudpickle
```

The two central classes in Kafi Streams are *TopologyNode* and *Streams".

## TopologyNode

The *TopologyNode* class is the fluent API on top of [*pydbsp*](https://github.com/brurucy/pydbsp) by Bruno Rucy, and is heavily inspired by the Kafka Streams DSL.

The class is completely abstracted away from Kafka. It does not know anything about Kafka. It just receives inputs, processes them relationally using pydbsp, and returns the outputs. This is why it can serve also as a test harness similar to the "TopologyTestDriver" in Kafka Streams.

These are the dependencies of the TopologyNode class, from bottom to top.

### pydbsp

The by far most important building block is pydbsp by Bruno Rucy. It is the heart of Kafi Streams. It is the actual stream processing engine.

### msgpack

In pydbsp, the fundamental data type is the *ZSet*. ZSets are implemented as dictionaries in pydbsp, where the keys are rows and the values are weights (=integers), e.g.:
```python
{"row_1": 1, "row_2": 0, "row_3": -1}
```

As Kafi Streams is typically used on top of Kafka where the payloads are encoded in JSON (and Kafi converts the JSONs into Python dictionaries automatically), I needed a fast way to serialize these dictionaries into a hashable form and deserialize them back to dictionaries.

This is the task of msgpack.

### cloudpickle

cloudpickle is used for serializing/deserializing the global state of the topology (technically, the state of the pydbsp `evaluator`) since the built-in Python pickler is unable to serialize it.

## Streams

The *Streams* subclass of *TopologyNode* adds support for Kafka.

### Kafi

Kafi (the "old" part) provides all the Kafka support for Kafi Streams. It continuously consumes source topics, pushes the data to pydbsp, gets the outputs and produces them to sink topics.

Kafi also provides chunking/dechunking support which is required for checkpointing - where the checkpoints can go either to real Kafka or, through Kafi's "Kafka emulation", also to disk, S3 or Azure Blob Storage.
